# 03 — Train Query Analyzer (~2–3 hours on Colab T4)

Fine-tunes DistilBERT multitask heads:
- retrieval necessity (binary)
- temporal class (current / historical / timeless)
- confidence

Defaults: 6–8 epochs, batch 16, lr 2e-5. If runtime exceeds ~3h, set `FREEZE_LAYERS=4`.

In [ ]:
import sys, torch
from pathlib import Path

ROOT = Path('..').resolve()
if not (ROOT / 'src').exists():
    ROOT = Path('.').resolve()
sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
EPOCHS = 6          # raise to 8 if training finishes early
BATCH_SIZE = 16
LR = 2e-5
FREEZE_LAYERS = 0   # set 4 to speed up if needed

from src.data_schema import load_queries
from src.query_analyzer import train_analyzer

queries = load_queries(ROOT / 'data' / 'queries.jsonl')
train = [q for q in queries if q.split == 'train']
val = [q for q in queries if q.split == 'val']
print(len(train), 'train /', len(val), 'val')

result = train_analyzer(
    train_texts=[q.query for q in train],
    train_needs=[q.needs_retrieval for q in train],
    train_temporal=[q.temporal_class for q in train],
    val_texts=[q.query for q in val],
    val_needs=[q.needs_retrieval for q in val],
    val_temporal=[q.temporal_class for q in val],
    artifacts_dir=ROOT / 'artifacts',
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    freeze_layers=FREEZE_LAYERS,
)
print('Best val score:', result['best_val_score'])
result['history']

In [ ]:
from src.query_analyzer import QueryAnalyzerService

svc = QueryAnalyzerService.from_pretrained_bundle(ROOT / 'artifacts')
for q in [
    'Who is the current CEO of Apple?',
    'Who was the Prime Minister of the United Kingdom in 2022?',
    'Define entropy.',
    'What is 2 + 2?',
]:
    print(q, '->', svc.predict(q))